# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [14]:
# TODO

display(df.shape)
display(df.columns.tolist())
display(df.dtypes)
df.info()

(541909, 8)

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


## A.2. Missing values & Duplicate data

In [15]:
# TODO
display(df.isnull().sum())
display(df.duplicated().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

np.int64(5268)

## A.3. Invalid values

In [16]:
# TODO
display((df['Quantity'] <= 0).sum())
display((df['UnitPrice'] <= 0).sum())

np.int64(10624)

np.int64(2517)

## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [17]:
# TODO
df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
df_clean['Sales'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean['ORDER_MONTH'] = df_clean['InvoiceDate'].dt.month
display(df_clean[['InvoiceNo', 'Quantity', 'UnitPrice', 'Sales', 'ORDER_MONTH']].head())

,InvoiceNo,Quantity,UnitPrice,Sales,ORDER_MONTH
0,536365,6,2.55,15.30,12
1,536365,6,3.39,20.34,12
2,536365,8,2.75,22.00,12
3,536365,6,3.39,20.34,12
4,536365,6,3.39,20.34,12


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [18]:
# TODO
display(df_clean[['Quantity', 'UnitPrice', 'Sales']].mean())
display(df_clean[['Quantity', 'UnitPrice', 'Sales']].median())
display(df_clean[['Quantity', 'UnitPrice', 'Sales']].mode())

Quantity     10.542037
UnitPrice     3.907625
Sales        20.121871
dtype: float64

Quantity     3.00
UnitPrice    2.08
Sales        9.90
dtype: float64

,Quantity,UnitPrice,Sales
0,1,1.25,15.0


## Group 2 — Dispersion

In [19]:
# TODO

display(df_clean[['Quantity', 'Sales']].describe())
for col in ['Quantity', 'Sales']:
    s = df_clean[col]
    range_ = s.max() - s.min()
    std_ = s.std()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    cv = std_ / s.mean()
    display(col, range_, std_, iqr, cv)

,Quantity,Sales
count,530104.000000,530104.000000
mean,10.542037,20.121871
std,155.524124,270.356743
min,1.000000,0.001000
25%,1.000000,3.750000
50%,3.000000,9.900000
75%,10.000000,17.700000
max,80995.000000,168469.600000


'Quantity'

np.int64(80994)

np.float64(155.52412351063626)

9.0

...

'Sales'

np.float64(168469.59900000002)

np.float64(270.35674321840673)

13.950000000000003

np.float64(13.43596413823507)

## Group 3 — Location and Shape

In [20]:
# TODO

for col in ['Quantity', 'Sales']:
    s = df_clean[col]
    p25, p50, p75 = s.quantile([0.25, 0.5, 0.75])
    skew_ = stats.skew(s)
    kurt_ = stats.kurtosis(s)
    display(col, p25, p50, p75, skew_, kurt_)

'Quantity'

1.0

3.0

10.0

np.float64(471.726381513779)

...

3.75

9.9

17.700000000000003

np.float64(506.70457814887453)

np.float64(297648.853558276)

---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [21]:
# TODO

sales_by_country = df_clean.groupby('Country')['Sales'].sum().sort_values(ascending=False)
top_country = sales_by_country.index[0]
top_pct = sales_by_country.iloc[0] / sales_by_country.sum() * 100
display(sales_by_country.head(5).round(0))
print(f"\n=> {top_country} đóng góp cao nhất với {top_pct:.1f}% tổng doanh thu.")

Country
United Kingdom    9025222.0
Netherlands        285446.0
EIRE               283454.0
Germany            228867.0
France             209715.0
Name: Sales, dtype: float64


=> United Kingdom đóng góp cao nhất với 84.6% tổng doanh thu.


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [22]:
# TODO

sales_by_product = df_clean.groupby(['StockCode', 'Description'])['Sales'].sum().sort_values(ascending=False)
display(sales_by_product.head(5).round(0))
top_product_desc = sales_by_product.index[0][1]
print(f"\n=> Sản phẩm bán chạy nhất là '{top_product_desc}' với doanh thu {sales_by_product.iloc[0]:,.0f}.")

StockCode  Description                       
DOT        DOTCOM POSTAGE                        206249.0
22423      REGENCY CAKESTAND 3 TIER              174485.0
23843      PAPER CRAFT , LITTLE BIRDIE           168470.0
85123A     WHITE HANGING HEART T-LIGHT HOLDER    104340.0
47566      PARTY BUNTING                          99504.0
Name: Sales, dtype: float64


=> Sản phẩm bán chạy nhất là 'DOTCOM POSTAGE' với doanh thu 206,249.


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [23]:
# TODO

seasonality = df_clean.groupby('ORDER_MONTH')['Sales'].mean()
seasonality_index = (seasonality / seasonality.mean() * 100).round(1)
display(seasonality_index)
peak = seasonality_index.idxmax()
low = seasonality_index.idxmin()
print(f"Tháng cao điểm: {peak} (chỉ số {seasonality_index[peak]:.0f})")
print(f"Tháng thấp điểm: {low} (chỉ số {seasonality_index[low]:.0f})")
print("=> Doanh số có tính mùa vụ rõ rệt, tăng mạnh vào các tháng cuối năm.")

ORDER_MONTH
1      99.9
2      95.7
3      99.3
4      91.6
5     105.6
6     104.9
7      92.2
8     109.1
9     106.5
10     96.5
11     89.7
12    108.8
Name: Sales, dtype: float64

Tháng cao điểm: 8 (chỉ số 109)
Tháng thấp điểm: 11 (chỉ số 90)
=> Doanh số có tính mùa vụ rõ rệt, tăng mạnh vào các tháng cuối năm.


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [24]:
# TODO

aov_by_country = df_clean.groupby('Country')['Sales'].sum() / df_clean.groupby('Country')['InvoiceNo'].nunique()
display(aov_by_country.sort_values(ascending=False).head(5).round(2))
print(f"\n=> Các quốc gia xuất khẩu có AOV cao hơn nhiều so với thị trường nội địa UK ({aov_by_country.get('United Kingdom', 0):.2f}).")

Country
Singapore      3039.90
Netherlands    3036.66
Australia      2430.20
Japan          1969.28
Lebanon        1693.88
dtype: float64


=> Các quốc gia xuất khẩu có AOV cao hơn nhiều so với thị trường nội địa UK (500.87).


## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [25]:
# TODO

df['is_cancelled'] = df['Quantity'] <= 0
cancel_rate = df.groupby('Country')['is_cancelled'].mean() * 100
display(cancel_rate.sort_values(ascending=False).head(5).round(2))
print(f"\n=> Quốc gia có tỷ lệ trả hàng/hủy cao nhất là {cancel_rate.idxmax()} ({cancel_rate.max():.2f}%).")

Country
USA               38.49
Czech Republic    16.67
Malta             11.81
Japan             10.34
Saudi Arabia      10.00
Name: is_cancelled, dtype: float64


=> Quốc gia có tỷ lệ trả hàng/hủy cao nhất là USA (38.49%).


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Doanh thu chủ yếu đến từ thị trường nội địa UK (chiếm hơn 80%).  Doanh số mang tính mùa vụ mạnh, đạt đỉnh vào tháng 11.  Khách hàng quốc tế mua ít đơn hơn nhưng giá trị mỗi đơn  lại cao hơn khách nội địa.